# FD001 Window And Cap Search

Objetivo: buscar `window_size` y `rul_cap` usando solo validación artificial. No se usa el test oficial FD001.


In [ ]:
from collections import OrderedDict
from pathlib import Path
import sys
import warnings

import numpy as np
import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RESULTS_DIR = PROJECT_ROOT / "results"
FIGURES_DIR = PROJECT_ROOT / "figures"
PREDICTIONS_DIR = PROJECT_ROOT / "predictions"
for path in [RESULTS_DIR, FIGURES_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

SEED = 42
DATA_DIR = PROJECT_ROOT / "CMAPSSData"
EVAL_SIZE = 0.2
CUT_RULS = (20, 50, 80, 110, 140)
DEFAULT_WINDOW_SIZE = 30
DEFAULT_RUL_CAP = 125

pd.set_option("display.max_columns", 80)
pd.set_option("display.width", 140)


In [ ]:
from src.preprocessed_FD001 import (
    prepare_fd001_temporal_validation_only,
    preprocessing_summary,
)
from src.fd001_modeling import (
    metrics_by_model,
    metrics_by_rul_bin,
    plot_validation_diagnostics,
    prediction_frame,
    regression_metrics,
    set_global_seed,
)


def available_tabular_factories(random_state=42):
    from sklearn.ensemble import (
        ExtraTreesRegressor,
        GradientBoostingRegressor,
        HistGradientBoostingRegressor,
        RandomForestRegressor,
    )

    factories = OrderedDict()
    availability = []
    has_external_boosting = False

    factories["RandomForestRegressor"] = lambda: RandomForestRegressor(
        n_estimators=250,
        max_depth=14,
        min_samples_leaf=3,
        random_state=random_state,
        n_jobs=-1,
    )

    try:
        from xgboost import XGBRegressor
        factories["XGBRegressor"] = lambda: XGBRegressor(
            n_estimators=160,
            max_depth=3,
            learning_rate=0.04,
            subsample=0.85,
            colsample_bytree=0.85,
            objective="reg:squarederror",
            random_state=random_state,
            n_jobs=-1,
            tree_method="hist",
            verbosity=0,
        )
        has_external_boosting = True
        availability.append("XGBoost disponible: se incluye XGBRegressor.")
    except Exception as exc:
        availability.append(f"XGBoost no disponible: {type(exc).__name__}.")

    try:
        from lightgbm import LGBMRegressor
        factories["LGBMRegressor"] = lambda: LGBMRegressor(
            n_estimators=220,
            max_depth=-1,
            num_leaves=31,
            learning_rate=0.035,
            subsample=0.85,
            colsample_bytree=0.85,
            reg_lambda=0.1,
            random_state=random_state,
            n_jobs=-1,
            verbose=-1,
        )
        has_external_boosting = True
        availability.append("LightGBM disponible: se incluye LGBMRegressor.")
    except Exception as exc:
        availability.append(f"LightGBM no disponible: {type(exc).__name__}.")

    if not has_external_boosting:
        availability.append("Sin XGBoost/LightGBM: se usan fallbacks sklearn.")
        factories["HistGradientBoostingRegressor"] = lambda: HistGradientBoostingRegressor(
            max_iter=180,
            learning_rate=0.05,
            max_leaf_nodes=31,
            l2_regularization=0.01,
            random_state=random_state,
        )
        factories["GradientBoostingRegressor"] = lambda: GradientBoostingRegressor(
            n_estimators=160,
            learning_rate=0.05,
            max_depth=3,
            subsample=0.85,
            random_state=random_state,
        )
        factories["ExtraTreesRegressor"] = lambda: ExtraTreesRegressor(
            n_estimators=220,
            max_depth=16,
            min_samples_leaf=2,
            random_state=random_state,
            n_jobs=-1,
        )
    return factories, availability

def fit_predict_model(prepared, model_name, model, representation, sample_weight=None):
    if sample_weight is None:
        model.fit(prepared["X_train"], prepared["y_train"])
    else:
        model.fit(prepared["X_train"], prepared["y_train"], sample_weight=sample_weight)
    preds = model.predict(prepared["X_eval"])
    return prediction_frame(
        prepared["eval_df"],
        preds,
        model_name=model_name,
        representation=representation,
    ), model


In [ ]:
def choose_best_boosting_name():
    factories, availability = available_tabular_factories(random_state=SEED)
    metrics_path = RESULTS_DIR / "fd001_boosting_metrics.csv"
    if metrics_path.exists():
        previous = pd.read_csv(metrics_path)
        candidates = previous.loc[previous["model_name"] != "RandomForestRegressor"].copy()
        if not candidates.empty:
            return candidates.sort_values(["rmse", "mae"]).iloc[0]["model_name"], availability
    for candidate in ["XGBRegressor", "LGBMRegressor", "HistGradientBoostingRegressor", "GradientBoostingRegressor", "ExtraTreesRegressor"]:
        if candidate in factories:
            return candidate, availability
    raise RuntimeError("No hay boosting disponible.")


def add_bin_values(row, bin_metrics):
    for label in ["0-30", "30-60", "60-90", "90+"]:
        safe_label = label.replace("-", "_").replace("+", "plus")
        subset = bin_metrics.loc[bin_metrics["rul_bin"].astype(str) == label]
        if subset.empty:
            row[f"mae_rul_{safe_label}"] = np.nan
            row[f"dangerous_error_pct_rul_{safe_label}"] = np.nan
        else:
            row[f"mae_rul_{safe_label}"] = subset.iloc[0]["mae"]
            row[f"dangerous_error_pct_rul_{safe_label}"] = subset.iloc[0]["dangerous_error_pct"]
    return row


## Configuración

Se compara RandomForest contra el mejor boosting disponible. La elección del boosting puede venir del notebook 09 si ya fue ejecutado; si no, se elige el primer candidato instalado.


In [ ]:
WINDOW_SIZES = [10, 20, 30, 40, 50, 60]
RUL_CAPS = [100, 125, 150, None]

model_factories, availability_notes = available_tabular_factories(random_state=SEED)
best_boosting_name, availability_notes = choose_best_boosting_name()
print("\n".join(availability_notes))
print(f"Boosting elegido para la búsqueda: {best_boosting_name}")


## Búsqueda controlada


In [ ]:
rows = []

for window_size in WINDOW_SIZES:
    print(f"Preparando features temporales para window={window_size}...")
    base_prepared = prepare_fd001_temporal_validation_only(
        data_dir=DATA_DIR,
        eval_size=EVAL_SIZE,
        random_state=SEED,
        max_rul=None,
        cut_ruls=CUT_RULS,
        window_size=window_size,
    )
    representation = f"temporal_w{window_size}"

    for rul_cap in RUL_CAPS:
        print(f"Evaluando window={window_size}, rul_cap={rul_cap}...")
        prepared = dict(base_prepared)
        if rul_cap is None:
            prepared["y_train"] = base_prepared["y_train_raw"].copy()
        else:
            prepared["y_train"] = base_prepared["y_train_raw"].clip(upper=rul_cap).copy()

        candidates = OrderedDict([
            ("RandomForestRegressor", model_factories["RandomForestRegressor"]),
            (best_boosting_name, model_factories[best_boosting_name]),
        ])
        for model_name, factory in candidates.items():
            print(f"  Entrenando {model_name}")
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                pred_df, _ = fit_predict_model(prepared, model_name, factory(), representation)
            metric_row = metrics_by_model(pred_df).iloc[0].to_dict()
            bin_metrics = metrics_by_rul_bin(pred_df)
            metric_row.update({
                "window_size": window_size,
                "rul_cap": "None" if rul_cap is None else rul_cap,
                "n_features": len(prepared["feature_columns"]),
                "target_used_for_training": "RUL_raw" if rul_cap is None else "RUL_capped",
            })
            metric_row = add_bin_values(metric_row, bin_metrics)
            rows.append(metric_row)

search_results = pd.DataFrame(rows)
rank_cols = ["rmse", "mae", "cmapss_score_mean", "dangerous_error_pct", "mae_rul_0_30", "mae_rul_30_60", "mae_rul_60_90"]
for col in rank_cols:
    search_results[f"rank_{col}"] = search_results[col].rank(method="min")
search_results["balanced_validation_rank"] = (
    search_results["rank_rmse"]
    + search_results["rank_mae"]
    + search_results["rank_cmapss_score_mean"]
    + 1.5 * search_results["rank_dangerous_error_pct"]
    + 1.5 * search_results["rank_mae_rul_0_30"]
    + 0.75 * search_results["rank_mae_rul_30_60"]
    + 0.5 * search_results["rank_mae_rul_60_90"]
)
search_results = search_results.sort_values(["balanced_validation_rank", "rmse", "dangerous_error_pct"]).reset_index(drop=True)
search_results.to_csv(RESULTS_DIR / "fd001_window_cap_search.csv", index=False)
search_results.head(15)


## Criterio de lectura

El ranking balanceado no reemplaza el juicio del modelo: combina error global, score asimétrico, errores peligrosos y MAE cerca de falla. No está elegido solo por RMSE.


In [ ]:
display(search_results.head(20))
print(RESULTS_DIR / "fd001_window_cap_search.csv")
